In [1]:
#data = 'https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv'
#!wget $data

In [30]:
import numpy as np
import pandas as pd

In [31]:
df = pd.read_csv('car_fuel_efficiency.csv')
col = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year', 'fuel_efficiency_mpg']
df = df[col]
df.head()

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,170,159.0,3413.433759,2003,13.231729
1,130,97.0,3149.664934,2007,13.688217
2,170,78.0,3079.038997,2018,14.246341
3,220,NaN,2542.392402,2009,16.912736
4,210,140.0,3460.870990,2009,12.488369


EDA
1. There's one column with missing values. What is it? b. 'horsepower'
2. What's the median (50% percentile) for variable 'horsepower'? c. 149

In [32]:
df.isnull().sum()

engine_displacement      0
horsepower             708
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

In [33]:
df.horsepower.median()

np.float64(149.0)

Question 3
1. We need to deal with missing values for the column from Q1.
2. We have two options: fill it with 0 or with the mean of this variable.
3. Try both options. For each, train a linear regression model without regularization using the code from the lessons.
4. For computing the mean, use the training only!
5. Use the validation dataset to evaluate the models and compare the RMSE of each option.
6. Round the RMSE scores to 2 decimal digits using round(score, 2)
7. Which option gives better RMSE? b. With mean

In [60]:
def prepare_X(df, mean=True):
    X = df.copy()
    if mean:
        X = X.fillna(X.horsepower.mean())
    else:
        X = X.fillna(0)
    
    X = X.values
    return X
    
def train_linear_regression_reg(X, y, r=0.01):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    reg = r * np.eye(XTX.shape[0])
    XTX = XTX + reg

    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)
    
    return w[0], w[1:]

def rmse(y, z):
    e = y - z
    se = e ** 2
    mse = se.mean()
    rmse = mse ** 0.5
    return rmse

In [41]:
n = len(df)
n_val = n_test = int(0.2 * n)
n_train = n - (n_val + n_test)

np.random.seed(0)
idx = np.arange(n)
np.random.shuffle(idx)

df_shuffled = df.iloc[idx]

df_train = df_shuffled.iloc[:n_train].copy()
df_val = df_shuffled.iloc[n_train:n_train+n_val].copy()
df_test = df_shuffled.iloc[n_train+n_val:].copy()

y_train_orig = df_train.fuel_efficiency_mpg.values
y_val_orig = df_val.fuel_efficiency_mpg.values
y_test_orig = df_test.fuel_efficiency_mpg.values

y_train = np.log1p(df_train.fuel_efficiency_mpg.values)
y_val = np.log1p(df_val.fuel_efficiency_mpg.values)
y_test = np.log1p(df_test.fuel_efficiency_mpg.values)

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [71]:
X_train = prepare_X(df_train, True)
w0, w = train_linear_regression_reg(X_train, y_train, 0)
X_valid = prepare_X(df_val, True)
y_pred = X_valid @ w + w0
print(rmse(y_pred, y_val)) # With mean

0.03480336033253816


In [72]:
X_train = prepare_X(df_train, False)
w0, w = train_linear_regression_reg(X_train, y_train, 0)
X_valid = prepare_X(df_val, False)
y_pred = X_valid @ w + w0
print(rmse(y_pred, y_val)) # With 0

0.03801775537087802


Question 4
1. Now let's train a regularized linear regression.
2. For this question, fill the NAs with 0.
3. Try different values of r from this list: [0, 0.01, 0.1, 1, 5, 10, 100].
4. Use RMSE to evaluate the model on the validation dataset.
5. Round the RMSE scores to 2 decimal digits.
6. Which r gives the best RMSE?
7. If multiple options give the same best RMSE, select the smallest r. a. 0

In [73]:
X_train = prepare_X(df_train, False)
for i in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression_reg(X_train, y_train, i)
    X_valid = prepare_X(df_val, False)
    y_pred = X_valid @ w + w0
    print(i, rmse(y_pred, y_val))

0 0.03801775537087802
0.01 0.03803355743792981
0.1 0.03860102760899764
1 0.03944915172540636
5 0.03960033040235085
10 0.039620841222669095
100 0.03963963498841045


Question 5
1. We used seed 42 for splitting the data. Let's find out how selecting the seed influences our score.
2. Try different seed values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9].
3. For each seed, do the train/validation/test split with 60%/20%/20% distribution.
4. Fill the missing values with 0 and train a model without regularization.
5. For each seed, evaluate the model on the validation dataset and collect the RMSE scores.
6. What's the standard deviation of all the scores? To compute the standard deviation, use np.std.
7. Round the result to 3 decimal digits (round(std, 3))
8. What's the value of std? a. 0.001

In [106]:
rmse_all = np.array([])
for i in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    n = len(df)
    n_val = n_test = int(0.2 * n)
    n_train = n - (n_val + n_test)
    
    np.random.seed(i)
    idx = np.arange(n)
    np.random.shuffle(idx)
    
    df_shuffled = df.iloc[idx]
    
    df_train = df_shuffled.iloc[:n_train].copy()
    df_val = df_shuffled.iloc[n_train:n_train+n_val].copy()
    df_test = df_shuffled.iloc[n_train+n_val:].copy()
    
    y_train_orig = df_train.fuel_efficiency_mpg.values
    y_val_orig = df_val.fuel_efficiency_mpg.values
    y_test_orig = df_test.fuel_efficiency_mpg.values
    
    y_train = np.log1p(df_train.fuel_efficiency_mpg.values)
    y_val = np.log1p(df_val.fuel_efficiency_mpg.values)
    y_test = np.log1p(df_test.fuel_efficiency_mpg.values)
    
    del df_train['fuel_efficiency_mpg']
    del df_val['fuel_efficiency_mpg']
    del df_test['fuel_efficiency_mpg']
    
    X_train = prepare_X(df_train, False)
    w0, w = train_linear_regression_reg(X_train, y_train, 0)
    X_valid = prepare_X(df_val, False)
    y_pred = X_valid @ w + w0
    print(i, rmse(y_pred, y_val))
    rmse_all = np.append(rmse_all, rmse(y_pred, y_val))

print(round(np.std(rmse_all), 3))

0 0.03801775537087802
1 0.0392788533393729
2 0.039446530525595505
3 0.0387276370388226
4 0.03727535850105433
5 0.039384388339929154
6 0.03890763931346167
7 0.038379716269399816
8 0.040189869753460644
9 0.038607646441269794
0.001


Question 6
1. Split the dataset like previously, use seed 9.
2. Combine train and validation datasets.
3. Fill the missing values with 0 and train a model with r=0.001.
4. What's the RMSE on the test dataset? b. 0.515

In [88]:
n = len(df)
n_val = n_test = int(0.2 * n)
n_train = n - (n_val + n_test)

np.random.seed(9)
idx = np.arange(n)
np.random.shuffle(idx)

df_shuffled = df.iloc[idx]

df_train = df_shuffled.iloc[:n_train].copy()
df_val = df_shuffled.iloc[n_train:n_train+n_val].copy()
df_test = df_shuffled.iloc[n_train+n_val:].copy()

y_train_orig = df_train.fuel_efficiency_mpg.values
y_val_orig = df_val.fuel_efficiency_mpg.values
y_test_orig = df_test.fuel_efficiency_mpg.values

y_train = np.log1p(df_train.fuel_efficiency_mpg.values)
y_val = np.log1p(df_val.fuel_efficiency_mpg.values)
y_test = np.log1p(df_test.fuel_efficiency_mpg.values)

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [102]:
df_full_train = pd.concat((df_train, df_val))
y_full_train = np.append(y_train, y_val)

X_full_train = prepare_X(df_full_train, False)
w0, w = train_linear_regression_reg(X_full_train, y_full_train, 0.001)
X_test = prepare_X(df_test, False)
y_pred = X_test @ w + w0
print(rmse(y_pred, y_test))

0.039196136444546614


If your answer doesn't match options exactly, select the closest one. If the answer is exactly in between two options, select the higher value.